# 10 - Transaction Costs And Position Sizing

This notebook upgrades the strategy evaluation from a simple long/cash signal to a more realistic backtest.

It adds:

- Transaction costs
- Turnover
- Volatility-based position sizing
- Maximum position caps
- Per-asset strategy comparison

Input: `../data/processed/all_asset_predictions.csv`  
Output: cost-aware strategy metrics and position-level predictions.

## Important Terms

- **Transaction cost**: cost paid when changing position, usually measured in basis points.
- **Basis point**: 1 bp = 0.01%. So 10 bps = 0.10%.
- **Turnover**: absolute change in position from one day to the next.
- **Position sizing**: deciding how much capital to allocate.
- **Volatility targeting**: reduce position size when volatility is high.
- **Max exposure**: cap on position size, such as 100% long.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
DATA_DIR = Path("../data/processed")

## 1. Load All-Asset Predictions

In [2]:
predictions = pd.read_csv(DATA_DIR / "all_asset_predictions.csv", parse_dates=["Date"])
predictions = predictions.sort_values(["model_family", "ticker", "Date"]).reset_index(drop=True)

print(predictions.shape)
print(predictions["model_family"].value_counts())
print(predictions["ticker"].value_counts())
predictions.head()

(2508, 11)
model_family
HistGradientBoosting    627
LightGBM                627
Random Forest           627
XGBoost                 627
Name: count, dtype: int64
ticker
^GSPC    1008
^IXIC    1008
^NSEI     492
Name: count, dtype: int64


,Date,ticker,Close,regime_label,target_return_1d,target_direction_1d,model_family,fold,baseline_up_prob,hard_moe_up_prob,soft_moe_up_prob
0,2019-04-30,^GSPC,2945.830078,Bear,-0.007502,0,HistGradientBoosting,1,0.361960,0.559977,0.559962
1,2019-05-01,^GSPC,2923.729980,Bear,-0.002124,0,HistGradientBoosting,1,0.260570,0.389841,0.389841
2,2019-05-02,^GSPC,2917.520020,Bear,0.009638,1,HistGradientBoosting,1,0.681935,0.667915,0.667915
3,2019-05-03,^GSPC,2945.639893,Bear,-0.004471,0,HistGradientBoosting,1,0.650238,0.849816,0.849816
4,2019-05-06,^GSPC,2932.469971,Bear,-0.016512,0,HistGradientBoosting,1,0.276834,0.226250,0.226250


## 2. Choose Best Model

From notebook 08, the best overall model was Random Forest Soft-Gated MoE.

In [3]:
MODEL_FAMILY = "Random Forest"
PROB_COL = "soft_moe_up_prob"
THRESHOLD = 0.45

model_df = predictions[predictions["model_family"] == MODEL_FAMILY].copy()
model_df = model_df.sort_values(["ticker", "Date"]).reset_index(drop=True)

print(model_df.shape)
model_df.head()

(627, 11)


,Date,ticker,Close,regime_label,target_return_1d,target_direction_1d,model_family,fold,baseline_up_prob,hard_moe_up_prob,soft_moe_up_prob
0,2019-04-30,^GSPC,2945.830078,Bear,-0.007502,0,Random Forest,1,0.508347,0.509312,0.509312
1,2019-05-01,^GSPC,2923.729980,Bear,-0.002124,0,Random Forest,1,0.455550,0.413391,0.413391
2,2019-05-02,^GSPC,2917.520020,Bear,0.009638,1,Random Forest,1,0.506828,0.502453,0.502453
3,2019-05-03,^GSPC,2945.639893,Bear,-0.004471,0,Random Forest,1,0.607065,0.583852,0.583852
4,2019-05-06,^GSPC,2932.469971,Bear,-0.016512,0,Random Forest,1,0.430200,0.408780,0.408780


## 3. Build Cost-Aware Strategy Functions

In [4]:
def annualized_sharpe(returns, periods_per_year=252):
    returns = pd.Series(returns).dropna()
    if returns.std() == 0:
        return 0
    return returns.mean() / returns.std() * np.sqrt(periods_per_year)


def max_drawdown(returns):
    returns = pd.Series(returns).fillna(0)
    equity = (1 + returns).cumprod()
    peak = equity.cummax()
    dd = equity / peak - 1
    return dd.min()


def build_positions(group, prob_col, threshold=0.45, target_vol=0.12, max_position=1.0):
    group = group.sort_values("Date").copy()

    group["raw_signal"] = (group[prob_col] >= threshold).astype(float)

    # Use realized 30-day volatility from target returns as a simple risk estimate.
    realized_vol = group["target_return_1d"].rolling(30).std() * np.sqrt(252)
    group["realized_vol_30d"] = realized_vol

    vol_scale = target_vol / realized_vol
    vol_scale = vol_scale.replace([np.inf, -np.inf], np.nan).fillna(0)
    vol_scale = vol_scale.clip(lower=0, upper=max_position)

    group["position"] = group["raw_signal"] * vol_scale
    group["position"] = group["position"].clip(0, max_position)

    return group


def apply_transaction_costs(group, cost_bps=10):
    group = group.sort_values("Date").copy()
    cost_rate = cost_bps / 10000

    group["previous_position"] = group["position"].shift(1).fillna(0)
    group["turnover"] = (group["position"] - group["previous_position"]).abs()
    group["transaction_cost"] = group["turnover"] * cost_rate

    group["gross_strategy_return"] = group["position"] * group["target_return_1d"]
    group["net_strategy_return"] = group["gross_strategy_return"] - group["transaction_cost"]
    group["buy_hold_return"] = group["target_return_1d"]

    return group


def summarize_strategy(group):
    net_returns = group["net_strategy_return"]
    gross_returns = group["gross_strategy_return"]
    buy_hold = group["buy_hold_return"]

    return pd.Series({
        "rows": len(group),
        "net_sharpe": annualized_sharpe(net_returns),
        "gross_sharpe": annualized_sharpe(gross_returns),
        "buy_hold_sharpe": annualized_sharpe(buy_hold),
        "net_max_drawdown": max_drawdown(net_returns),
        "buy_hold_max_drawdown": max_drawdown(buy_hold),
        "avg_daily_net_return": net_returns.mean(),
        "avg_position": group["position"].mean(),
        "avg_turnover": group["turnover"].mean(),
        "total_transaction_cost": group["transaction_cost"].sum(),
    })

## 4. Run Cost-Aware Backtest

Try different transaction cost assumptions.

Typical rough assumptions for liquid index strategies might be 5-20 bps, depending on execution, instrument, and market.

In [ ]:
cost_scenarios_bps = [0, 5, 10, 20]
target_vol = 0.12
max_position = 1.0

all_cost_metrics = []
all_position_outputs = []

for cost_bps in cost_scenarios_bps:
    temp_list = []
    for _, g in model_df.groupby("ticker"):
        g = build_positions(g, PROB_COL, THRESHOLD, target_vol, max_position)
        g = apply_transaction_costs(g, cost_bps)
        temp_list.append(g)

    temp = pd.concat(temp_list, ignore_index=True)

    temp["cost_bps"] = cost_bps
    all_position_outputs.append(temp)

    summary = temp.groupby("ticker").apply(summarize_strategy).reset_index()
    summary["cost_bps"] = cost_bps
    all_cost_metrics.append(summary)

cost_metrics = pd.concat(all_cost_metrics, ignore_index=True)
position_outputs = pd.concat(all_position_outputs, ignore_index=True)

cost_metrics.sort_values(["cost_bps", "net_sharpe"], ascending=[True, False])

KeyError: 'ticker'

## 5. Overall Portfolio View

This equal-weights the assets each day after applying individual position sizing.

In [ ]:
portfolio_metrics = []

for cost_bps in cost_scenarios_bps:
    temp = position_outputs[position_outputs["cost_bps"] == cost_bps].copy()

    daily_portfolio = (
        temp
        .groupby("Date")
        .agg(
            net_strategy_return=("net_strategy_return", "mean"),
            gross_strategy_return=("gross_strategy_return", "mean"),
            buy_hold_return=("buy_hold_return", "mean"),
            avg_position=("position", "mean"),
            avg_turnover=("turnover", "mean"),
            transaction_cost=("transaction_cost", "mean"),
        )
        .reset_index()
    )

    portfolio_metrics.append({
        "cost_bps": cost_bps,
        "net_sharpe": annualized_sharpe(daily_portfolio["net_strategy_return"]),
        "gross_sharpe": annualized_sharpe(daily_portfolio["gross_strategy_return"]),
        "buy_hold_sharpe": annualized_sharpe(daily_portfolio["buy_hold_return"]),
        "net_max_drawdown": max_drawdown(daily_portfolio["net_strategy_return"]),
        "buy_hold_max_drawdown": max_drawdown(daily_portfolio["buy_hold_return"]),
        "avg_daily_net_return": daily_portfolio["net_strategy_return"].mean(),
        "avg_position": daily_portfolio["avg_position"].mean(),
        "avg_turnover": daily_portfolio["avg_turnover"].mean(),
    })

portfolio_metrics = pd.DataFrame(portfolio_metrics)
portfolio_metrics

## 6. Save Outputs

In [ ]:
cost_metrics.to_csv(DATA_DIR / "transaction_cost_per_asset_metrics.csv", index=False)
portfolio_metrics.to_csv(DATA_DIR / "transaction_cost_portfolio_metrics.csv", index=False)
position_outputs.to_csv(DATA_DIR / "position_sized_predictions.csv", index=False)

print("Saved:", DATA_DIR / "transaction_cost_per_asset_metrics.csv")
print("Saved:", DATA_DIR / "transaction_cost_portfolio_metrics.csv")
print("Saved:", DATA_DIR / "position_sized_predictions.csv")

## 7. Plot Cost Sensitivity

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(portfolio_metrics["cost_bps"], portfolio_metrics["net_sharpe"], marker="o", label="Net Strategy")
plt.plot(portfolio_metrics["cost_bps"], portfolio_metrics["buy_hold_sharpe"], marker="o", label="Buy & Hold")
plt.title("Portfolio Sharpe Sensitivity To Transaction Costs")
plt.xlabel("Transaction Cost Assumption (bps)")
plt.ylabel("Sharpe Ratio")
plt.legend()
plt.show()

## 8. Plot Equity Curves At 10 bps

In [ ]:
selected_cost = 10
selected = position_outputs[position_outputs["cost_bps"] == selected_cost].copy()

for ticker in selected["ticker"].unique():
    temp = selected[selected["ticker"] == ticker].sort_values("Date")

    net_equity = (1 + temp["net_strategy_return"]).cumprod()
    buy_hold_equity = (1 + temp["buy_hold_return"]).cumprod()

    plt.figure(figsize=(12, 5))
    plt.plot(temp["Date"], buy_hold_equity, label="Buy & Hold", color="black")
    plt.plot(temp["Date"], net_equity, label="Position-Sized Net Strategy")
    plt.title(f"Cost-Aware Equity Curve: {ticker}, {selected_cost} bps")
    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.legend()
    plt.show()

## Interpretation

This notebook answers a practical question:

> Does the model still look useful after transaction costs and risk-based position sizing?

A strong final result is not just high accuracy. It should show reasonable turnover, controlled drawdown, and acceptable sensitivity to cost assumptions.